# Traffic Congestion Prediction Model

## Goal
The goal of this section is to train a machine learning model that can predict traffic speed based on engineered features.

This is an important step because:
- Traffic speed determines how congested a road is
- If we can predict speed, we can estimate travel time
- This allows us to choose the most efficient route

---

## What is a Model?

A model is a function that learns patterns from data and uses those patterns to make predictions.

In this project:
- Input (X): Features about the road, time, and traffic patterns
- Output (y): Traffic speed

The model learns:
"Given these conditions, what will the traffic speed be?"

---

## Datasets Used

We are primarily using the `congestion_ml` dataset for modeling.

### congestion_ml
This dataset contains:
- Engineered features (time, lag, rolling averages)
- Traffic-related metrics (volume, speed, travel time)
- Encoded categorical variables (borough, direction)

This is the dataset we use to TRAIN the model.

---

### routing_edges
This dataset represents the road network:
- Nodes (intersections)
- Edges (roads between nodes)
- Travel time and distance

This will be used later for routing.

---

### routing_nodes
This dataset contains:
- Node IDs
- Latitude and longitude

This helps map the road network spatially.

---

## Important
For this notebook, we only use `congestion_ml` for training the model.

In [21]:
# imports

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

In [23]:
## load dataset

df = pd.read_csv("../../data/ml_datasets/congestion_ml.csv")
df.head()

,SegmentID,lat,lon,avg_speed,avg_travel_time,avg_vol_hist,peak_vol_hist,std_vol_hist,peak_hour_hist,hour,...,borough_Brooklyn,borough_Manhattan,borough_Queens,borough_Staten Island,Direction_EB,Direction_NB,Direction_SB,Direction_WB,Vol,is_congested
0,-0.258833,-1.378228,-3.195527,1.689452,-0.567263,-0.604429,-0.443698,-0.518378,-2.255487,-1.652075,...,False,False,False,True,False,False,False,True,16.0,0
1,-0.258833,-1.378228,-3.195527,1.689452,-0.567263,-0.604429,-0.443698,-0.518378,-2.255487,-1.652075,...,False,False,False,True,False,False,False,True,19.0,0
2,-0.258833,-1.378228,-3.195527,1.689452,-0.567263,-0.604429,-0.443698,-0.518378,-2.255487,-1.652075,...,False,False,False,True,False,False,False,True,10.0,0
3,-0.258833,-1.378228,-3.195527,1.689452,-0.567263,-0.604429,-0.443698,-0.518378,-2.255487,-1.652075,...,False,False,False,True,False,False,False,True,12.0,0
4,-0.258833,-1.378228,-3.195527,1.689452,-0.567263,-0.604429,-0.443698,-0.518378,-2.255487,-1.507990,...,False,False,False,True,False,False,False,True,16.0,0


## Data Cleaning

Before training any models, we need to ensure the dataset is clean.

Feature engineering (such as lag and rolling features) often creates missing values (NaN). These occur because some rows do not have enough historical data.

We remove these rows so the model only learns from complete and valid data.

In [24]:
df = df.dropna()
print("Shape after cleaning:", df.shape)

Shape after cleaning: (123334, 34)


## Define Features (X) and Target (y)

We split the dataset into:

- Features (X): all columns used to make predictions  
- Target (y): the value we want to predict  

In this project, we predict **traffic volume (Vol)** — the number of vehicles passing a road segment at a given time. This is the direct measure of congestion.

We remove the target column and any columns derived from it from X to prevent data leakage.

In [25]:
# Feature Selection Mode (Choose ONE)

USE_LEAKAGE_SAFE_MODE = True  # change to False if you want simple version

if USE_LEAKAGE_SAFE_MODE:
    # clean ML version (recommended)
    drop_cols = [
        "Vol",             # target — excluded from features
        "is_congested",    # derived directly from Vol — would leak the target
        "avg_travel_time", # static per-link average, correlated with volume
    ]

    print("Using CLEAN MODE (recommended)")
else:
    # simple baseline version
    drop_cols = [
        "Vol",
        "is_congested",
    ]

    print("Using SIMPLE MODE")

X = df.drop(columns=drop_cols)
y = df["Vol"]

print("X shape:", X.shape)
print("y shape:", y.shape)

Using CLEAN MODE (recommended)
X shape: (123334, 31)
y shape: (123334,)


## Train/Test Split

We divide the data into:

- Training set (80%) → used to train the model  
- Testing set (20%) → used to evaluate the model  

This allows us to test how well the model performs on unseen data.

In [26]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

## Model 1: Linear Regression

Linear Regression is the simplest machine learning model.

It assumes that the relationship between features and the target is linear.

In this context, it tries to learn relationships like:
- higher traffic volume → lower speed  
- rush hour → slower traffic  

This model is useful as a baseline to compare more complex models against.

In [ ]:
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

lr_pred = lr_model.predict(X_test)

## Model 2: Decision Tree

A Decision Tree model splits the data into branches based on feature values.

It learns rules such as:
- if rush hour → predict lower speed  
- if weekend → predict higher speed  

This allows the model to capture non-linear patterns in traffic behavior.

In [ ]:
dt_model = DecisionTreeRegressor(max_depth=5, random_state=42)
dt_model.fit(X_train, y_train)

dt_pred = dt_model.predict(X_test)

## Model 3: Random Forest

Random Forest is an ensemble model made up of many decision trees.

Instead of relying on a single tree, it:
- builds multiple trees  
- averages their predictions  

This reduces overfitting and usually improves accuracy.

Random Forest is often one of the best-performing models for tabular datasets like this one.

In [27]:
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_test)

## Model Evaluation

We evaluate each model using:

- MAE (Mean Absolute Error): average difference between predicted and actual values  
- RMSE (Root Mean Squared Error): penalizes larger errors more  

Lower values indicate better performance.

In [30]:
def evaluate(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    return mae, rmse

results = {}
results["Random Forest"] = evaluate(y_test, rf_pred)
print(results["Random Forest"])

(13.388611231648854, np.float64(23.390938963985295))


NOTE: Run code below after both models are done. If not done yet, then you should use your corresponding results[model_type] after running all the cells above.

In [ ]:
results["Linear Regression"] = evaluate(y_test, lr_pred)
results["Decision Tree"] = evaluate(y_test, dt_pred)

for model, (mae, rmse) in results.items():
    print(f"{model} -> MAE: {mae:.4f}, RMSE: {rmse:.4f}")

## Model Comparison

We compare all models to determine which performs best.

The best model is the one with the lowest error.

This model will later be used in the routing algorithm to predict traffic speed and estimate travel time.

In [ ]:
pd.DataFrame(results, index=["MAE", "RMSE"]).T

## Final Interpretation

- Which model performed best?
- How much better was it compared to the others?
- Why might this model perform better?

In general:
- Linear Regression is simple but limited  
- Decision Trees capture more complex patterns  
- Random Forest combines multiple trees and often achieves the best performance  

The selected model will be used to predict traffic speeds, which will directly impact route optimization.